# Parte 3: motor de recomendações

> **Pergunta do case:** usando os insights das Partes 1 e 2, construa um sistema simples de recomendações que analisa uma campanha e gera sugestões de melhoria. Requisitos: (1) recebe uma linha do dataset e gera pelo menos 3 recomendações; (2) acionáveis e quantificadas; (3) contextuais (plataforma, categoria e outros atributos); (4) ordenadas por impacto estimado; (5) protótipo funcional.

**A ideia:** o Modelo A da Parte 2 prevê a nota de um criativo **antes** de postar. Então dá para perguntar a ele "e se este mesmo vídeo tivesse um gancho?": mudo só aquela coluna, peço a previsão de novo, e a diferença é o ganho estimado. É como um simulador de "e se". A Parte 1 completa com o efeito observado na métrica que importa para a campanha (CTR, engajamento ou retenção), **na rede dela**.

| Passo | O que mostra |
|---|---|
| 1 | De onde vem cada número de uma recomendação (por extenso, para uma campanha) |
| 2 | Quais mudanças o motor considera, e por que essas |
| 3 | O motor completo, conferido contra o passo a passo |
| 4 | O motor em campanhas que ele nunca viu (uma de cada rede) |
| 5 | É contextual? O mesmo vídeo em três redes |
| 6 | O motor faz sentido? Conferências em 100 campanhas |

O motor fica em `src/recomendacoes.py` e também roda pela linha de comando: `python -m src.recomendacoes KLK-0123`.

## Passo 0: preparação

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path("../..")
sys.path.append(str(RAIZ))
from src.features import COLUNAS_A
from src.limpeza import carregar, separar_treino_teste
from src.recomendacoes import METRICA_DO_OBJETIVO, carregar_motor, mudancas_possiveis, preparar, recomendar

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 230)

# O motor usa só o que foi aprendido no treino: o Modelo A (Parte 2) e a tabela de efeitos por rede (Parte 1)
motor = carregar_motor()
_, teste_bruto = separar_treino_teste(carregar(RAIZ / "data/raw/klike_challenge_dataset.csv"))
print(f"Modelo: {type(motor['modelo'][-1]).__name__} | efeitos da Parte 1: {len(motor['efeitos'])} linhas")
print("Formatos que cada rede usa com pelo menos 15 criativos:", motor["formatos_da_rede"])

Modelo: LinearRegression | efeitos da Parte 1: 72 linhas
Formatos que cada rede usa com pelo menos 15 criativos: {'LinkedIn': ['horizontal', 'quadrado'], 'Meta': ['horizontal', 'quadrado', 'vertical'], 'TikTok': ['quadrado', 'vertical']}


## Passo 1: de onde vem cada número de uma recomendação

Uma campanha do **teste** (criativos que o modelo nunca viu), tal como está no dataset original:

In [2]:
linha = teste_bruto[(teste_bruto["platform"] == "TikTok") & ~teste_bruto["has_hook"].astype(bool)].iloc[0]
campanha = preparar(linha, motor)  # a mesma limpeza das Partes 1 e 2
campanha[["campaign_id", "platform", "category", "objective", "has_hook", "has_face", "has_cta", "has_subtitle",
          "text_density", "format", "music_voice_ratio", "is_retargeting", "klike_score"]].to_frame("valor").T

,campaign_id,platform,category,objective,has_hook,has_face,has_cta,has_subtitle,text_density,format,music_voice_ratio,is_retargeting,klike_score
valor,KLK-0419,TikTok,Branding,awareness,0,1,1,sim,high,vertical,0.29,1,55.8


**(a) O ganho na nota, pelo modelo.** Prevejo a nota do criativo como ele é; depois copio a linha, ligo o gancho e prevejo de novo. Como tudo o resto fica igual, a diferença é o efeito do gancho **para este criativo** (no Modelo A, a música × TikTok faz o efeito depender da rede).

In [3]:
def prever(c):
    return float(np.clip(motor["modelo"].predict(pd.DataFrame([c])[COLUNAS_A])[0], 0, 100))


com_gancho = campanha.copy()
com_gancho["has_hook"] = 1
nota_hoje, nota_com_gancho = prever(campanha), prever(com_gancho)
print(f"Nota real: {campanha['klike_score']:.1f} | prevista hoje: {nota_hoje:.1f} | prevista com gancho: {nota_com_gancho:.1f} "
      f"| ganho: {nota_com_gancho - nota_hoje:+.1f} pontos")

Nota real: 55.8 | prevista hoje: 57.6 | prevista com gancho: 75.5 | ganho: +17.9 pontos

**(b) O efeito na métrica da campanha, pela Parte 1.** Cada objetivo quer mover uma métrica (Parte 1, pergunta 1.2): awareness → retenção, engagement → engajamento, traffic → CTR. Em conversions e app install, a Parte 1 mostrou que o criativo não mexe no custo por conversão; o efeito mostrado é no CTR, a etapa anterior. Procuro, na tabela `efeitos_por_plataforma.csv` (Parte 1, pergunta 1.3), a diferença das medianas com e sem gancho **nessa rede e nessa métrica**:

In [4]:
metrica = METRICA_DO_OBJETIVO[campanha["objective"]]
efeitos = motor["efeitos"]
linha_efeito = efeitos[(efeitos["plataforma"] == campanha["platform"]) & (efeitos["atributo"] == "Gancho (hook)")
                       & (efeitos["metrica"] == metrica)]
linha_efeito

,plataforma,atributo,metrica,mediana_com,mediana_sem,diferenca_pct,n_com,n_sem,intervalo_baixo_pct,intervalo_alto_pct,confiavel
26,TikTok,Gancho (hook),Retenção,0.559341,0.423077,32.207792,56,63,16.9656,73.940217,True


A recomendação junta os dois: "adicionar um gancho pode aumentar a nota em ~X pontos e o CTR em ~Y% no TikTok". O número da métrica só aparece se a coluna `confiavel` for verdadeira (o intervalo de 95% do bootstrap da Parte 1 fica de um lado do zero) e se os dois grupos tiverem pelo menos 15 criativos. Um efeito só "indício" pode até ter o sinal oposto ao da nota (por exemplo, o CTA sobe a nota, mas no engajamento do TikTok dá −11%), e confundiria quem lê.

## Passo 2: quais mudanças o motor considera?

Só o que o time controla e o que as Partes 1 e 2 sustentam:

| Mudança | Quando aparece | Base |
|---|---|---|
| adicionar gancho, rosto, CTA, legenda | o criativo não tem | os maiores efeitos da Parte 2 (+8 a +18 pontos) |
| texto na tela para "médio" | está em "pouco" ou "muito" | ponto ótimo no meio (Partes 1 e 2) |
| trocar música por voz | música/voz acima de 0,3 no TikTok, ou de 0,5 nas outras redes | só pesa no TikTok (Parte 2) |
| retargeting | a campanha não usa | +4 pontos na Parte 2; é decisão de público, não de vídeo |
| mudar o formato | a rede usa o outro formato com pelo menos 15 criativos | efeito pequeno; o TikTok não tem horizontal e o LinkedIn tem só 9 verticais |

**Fica de fora:**
- **a duração:** a Parte 1 não achou duração ideal, e na Parte 2 ela vale 0,02 ponto por segundo. Recomendar "reduza de 45 para 20 s" não teria base nestes dados;
- **trocar de plataforma, categoria ou objetivo:** não é melhorar a campanha, é fazer outra;
- **a idade do público:** a Parte 1 não achou padrão claro.

**Prioridade:** pela nota prevista, que é o alvo da Parte 2 e resume o funil. Mudanças que o modelo diz que **não** sobem a nota (por exemplo, trocar para um formato pior) são descartadas.

**E se o criativo já fizer quase tudo certo?** O motor não inventa melhoria sem base. Se sobrarem menos de 3, ele completa com o que o criativo **deve manter**, também quantificado ("manter o gancho: tirá-lo custaria ~18 pontos").

As mudanças possíveis para a campanha do passo 1:

In [5]:
pd.DataFrame(mudancas_possiveis(campanha, motor), columns=["o que fazer", "colunas alteradas", "atributo na Parte 1", "direção"])

,o que fazer,colunas alteradas,atributo na Parte 1,direção
0,Adicionar um gancho nos 3 primeiros segundos,{'has_hook': 1},Gancho (hook),adicionar
1,Mudar o texto na tela de 'muito' para 'médio',{'text_density': 'medium'},Muito texto,tirar
2,Mudar o formato de vertical para quadrado,{'format': 'quadrado'},NaN,NaN


## Passo 3: o motor completo

`recomendar` faz os passos 1 e 2 para todas as mudanças possíveis, descarta as que não sobem a nota, ordena pelo ganho e devolve as 3 primeiras. Confiro que o ganho do gancho é o mesmo calculado à mão no passo 1:

In [6]:
recomendacoes = recomendar(linha, motor)
gancho = recomendacoes[recomendacoes["recomendação"].str.startswith("Adicionar um gancho")]
assert np.isclose(gancho["impacto na nota"].iloc[0], nota_com_gancho - nota_hoje)
recomendacoes

,tipo,recomendação,impacto na nota
prioridade,,,
1,melhorar,"Adicionar um gancho nos 3 primeiros segundos pode aumentar a nota Klike em ~18 pontos (de 58 para 76) e a retenção em ~+32% no TikTok, com base nos padrões do dataset.",17.922319
2,melhorar,"Mudar o texto na tela de 'muito' para 'médio' pode aumentar a nota Klike em ~15 pontos (de 58 para 73) e a retenção em ~+21% no TikTok, com base nos padrões do dataset.",15.089107
3,manter,Manter o rosto humano: tirá-lo custaria ~12 pontos de nota (de 58 para 46).,11.528615


## Passo 4: o motor em campanhas que ele nunca viu

Uma campanha do teste de cada rede, sorteada entre as que têm nota real abaixo da mediana (as que mais precisam de ajuda):

In [7]:
mediana = teste_bruto["klike_score"].median()
for rede in ["Meta", "TikTok", "LinkedIn"]:
    exemplo = teste_bruto[(teste_bruto["platform"] == rede) & (teste_bruto["klike_score"] < mediana)].sample(1, random_state=7).iloc[0]
    print(f"\n{exemplo['campaign_id']} | {rede} | {exemplo['category']} | objetivo: {exemplo['objective']} | nota real: {exemplo['klike_score']:.1f}")
    for prioridade, r in recomendar(exemplo, motor).iterrows():
        print(f"  {prioridade}. [{r['tipo']}] {r['recomendação']}")


KLK-0339 | Meta | App Install | objetivo: app_install | nota real: 44.0


  1. [melhorar] Mostrar um rosto humano no vídeo pode aumentar a nota Klike em ~12 pontos (de 52 para 63), com base nos padrões do dataset.
  2. [melhorar] Colocar legenda pode aumentar a nota Klike em ~8 pontos (de 52 para 60), com base nos padrões do dataset.
  3. [melhorar] Mudar o texto na tela de 'pouco' para 'médio' pode aumentar a nota Klike em ~8 pontos (de 52 para 59), com base nos padrões do dataset.

KLK-0268 | TikTok | Lead Gen | objetivo: engagement | nota real: 47.0


  1. [melhorar] Adicionar um gancho nos 3 primeiros segundos pode aumentar a nota Klike em ~18 pontos (de 48 para 66), com base nos padrões do dataset.
  2. [melhorar] Mudar o texto na tela de 'muito' para 'médio' pode aumentar a nota Klike em ~15 pontos (de 48 para 63), com base nos padrões do dataset.
  3. [melhorar] Terminar com uma chamada para ação (CTA) pode aumentar a nota Klike em ~8 pontos (de 48 para 56), com base nos padrões do dataset.

KLK-0479 | LinkedIn | SaaS | objetivo: traffic | nota real: 53.0


  1. [melhorar] Adicionar um gancho nos 3 primeiros segundos pode aumentar a nota Klike em ~18 pontos (de 49 para 67) e o CTR em ~+133% no LinkedIn, com base nos padrões do dataset.
  2. [melhorar] Mostrar um rosto humano no vídeo pode aumentar a nota Klike em ~12 pontos (de 49 para 61), com base nos padrões do dataset.
  3. [melhorar] Rodar o criativo também para quem já conhece a marca (retargeting) pode aumentar a nota Klike em ~4 pontos (de 49 para 54), com base nos padrões do dataset.


**Lendo os exemplos:**

- **Cada campanha recebe 3 melhorias diferentes, conforme o que falta no criativo:** no Meta, rosto, legenda e texto; no TikTok, gancho, texto e CTA; no LinkedIn, gancho, rosto e retargeting.
- **A prioridade segue o tamanho do efeito:** gancho (+18 pontos), texto para "médio" (+8 a +15), rosto (+12), legenda e CTA (+8), retargeting (+4).
- **O efeito na métrica só aparece quando a Parte 1 tem um padrão confiável.** Na campanha de traffic do LinkedIn, o gancho vem com "+133% de CTR". Na de engagement do TikTok, nenhum efeito no engajamento é confiável, então só a nota aparece.

## Passo 5: é contextual? O mesmo vídeo em três redes

Pego um criativo do teste com muita música e pouca voz, sem gancho, e simulo que ele roda em cada rede (mudo só a plataforma; o formato vai para o mais comum da rede, porque o TikTok não tem horizontal):

In [8]:
base = teste_bruto[(teste_bruto["music_voice_ratio"] > 0.7) & ~teste_bruto["has_hook"].astype(bool)].iloc[0].copy()
FORMATO_MAIS_COMUM = {"Meta": "vertical", "TikTok": "vertical", "LinkedIn": "horizontal"}
for rede in ["Meta", "TikTok", "LinkedIn"]:
    versao = base.copy()
    versao["platform"], versao["format"] = rede, FORMATO_MAIS_COMUM[rede]
    print(f"\n{rede} (música/voz = {versao['music_voice_ratio']:.2f}, objetivo: {versao['objective']}):")
    for prioridade, r in recomendar(versao, motor).iterrows():
        print(f"  {prioridade}. [{r['tipo']}] {r['recomendação']}")


Meta (música/voz = 0.92, objetivo: conversions):


  1. [melhorar] Adicionar um gancho nos 3 primeiros segundos pode aumentar a nota Klike em ~18 pontos (de 55 para 73) e o CTR em ~+53% no Meta, com base nos padrões do dataset.
  2. [melhorar] Mudar o texto na tela de 'muito' para 'médio' pode aumentar a nota Klike em ~15 pontos (de 55 para 70) e o CTR em ~+47% no Meta, com base nos padrões do dataset.
  3. [melhorar] Rodar o criativo também para quem já conhece a marca (retargeting) pode aumentar a nota Klike em ~4 pontos (de 55 para 59), com base nos padrões do dataset.

TikTok (música/voz = 0.92, objetivo: conversions):


  1. [melhorar] Adicionar um gancho nos 3 primeiros segundos pode aumentar a nota Klike em ~18 pontos (de 51 para 69) e o CTR em ~+83% no TikTok, com base nos padrões do dataset.
  2. [melhorar] Mudar o texto na tela de 'muito' para 'médio' pode aumentar a nota Klike em ~15 pontos (de 51 para 66), com base nos padrões do dataset.
  3. [melhorar] Trocar parte da música por voz (música/voz de 0.9 para 0.3) pode aumentar a nota Klike em ~7 pontos (de 51 para 58), com base nos padrões do dataset.

LinkedIn (música/voz = 0.92, objetivo: conversions):
  1. [melhorar] Adicionar um gancho nos 3 primeiros segundos pode aumentar a nota Klike em ~18 pontos (de 49 para 67) e o CTR em ~+133% no LinkedIn, com base nos padrões do dataset.
  2. [melhorar] Mudar o texto na tela de 'muito' para 'médio' pode aumentar a nota Klike em ~15 pontos (de 49 para 64), com base nos padrões do dataset.
  3. [melhorar] Rodar o criativo também para quem já conhece a marca (retargeting) pode aumentar a nota Klike em 

**Sim, é contextual:**

- **Só no TikTok aparece "trocar música por voz"** (+7 pontos). Nas outras redes, a música não pesa (Parte 2), e o 3º lugar vai para o retargeting.
- **O efeito do gancho no CTR muda de rede para rede:** +53% no Meta, +83% no TikTok e +133% no LinkedIn. O de reduzir o texto (+47% de CTR) só aparece no Meta, a única rede em que ele é confiável.
- **A nota de partida também muda** (55, 51 e 49): a plataforma e o formato entram na previsão.
- **Categoria e objetivo:** o objetivo escolhe a métrica mostrada (aqui, conversions → CTR). A `category` não entra no modelo, porque repete o objetivo (notebook 2.1).

## Passo 6: o motor faz sentido?

Não dá para provar que as recomendações **causam** a melhora sem testar (um teste A/B). Mas dá para conferir se o motor se comporta como deveria, nas 100 campanhas do teste:

- **(a) Cobertura:** toda campanha recebe 3 recomendações? Quantas são melhorias e quantas são "manter"?
- **(b) Quais aparecem em primeiro?** Devem ser os maiores efeitos (gancho, rosto, texto).
- **(c) O ganho potencial bate com a nota real?** Se o motor está certo, as campanhas com mais espaço para melhorar (soma dos ganhos das 3 recomendações) devem ter, hoje, nota real **menor**. Meço com a correlação de Spearman (ordem).

In [9]:
linhas = []
for _, campanha_teste in teste_bruto.iterrows():
    r = recomendar(campanha_teste, motor)
    melhorias = r[r["tipo"] == "melhorar"]
    linhas.append({"campaign_id": campanha_teste["campaign_id"], "rede": campanha_teste["platform"], "nota real": campanha_teste["klike_score"],
                   "recomendações": len(r), "melhorias": len(melhorias), "ganho potencial": melhorias["impacto na nota"].sum(),
                   "primeira": r["recomendação"].iloc[0].split(" pode ")[0].split(":")[0]})
no_teste = pd.DataFrame(linhas)
print("Recomendações por campanha:", no_teste["recomendações"].value_counts().to_dict())
print("Melhorias por campanha:", no_teste["melhorias"].value_counts().sort_index().to_dict())
spearman = no_teste["ganho potencial"].corr(no_teste["nota real"], method="spearman")
print(f"Spearman entre ganho potencial e nota real: {spearman:.2f}")
no_teste["primeira"].value_counts().rename("vezes em 1º lugar").to_frame()

Recomendações por campanha: {3: 100}
Melhorias por campanha: {0: 1, 1: 3, 2: 15, 3: 81}
Spearman entre ganho potencial e nota real: -0.75


,vezes em 1º lugar
primeira,
Adicionar um gancho nos 3 primeiros segundos,50
Mostrar um rosto humano no vídeo,14
Mudar o texto na tela de 'muito' para 'médio',11
Colocar legenda,10
Rodar o criativo também para quem já conhece a marca (retargeting),6
Mudar o texto na tela de 'pouco' para 'médio',5
Terminar com uma chamada para ação (CTA),3
Manter o gancho nos 3 primeiros segundos,1


In [10]:
# Nota real por faixa de ganho potencial (4 faixas com o mesmo número de campanhas)
no_teste["faixa de ganho"] = pd.qcut(no_teste["ganho potencial"].rank(method="first"), 4,
                                     labels=["menor espaço", "2º quarto", "3º quarto", "maior espaço"])
no_teste.groupby("faixa de ganho", observed=True).agg(
    campanhas=("nota real", "size"), ganho_potencial_medio=("ganho potencial", "mean"), nota_real_mediana=("nota real", "median")).round(1)

,campanhas,ganho_potencial_medio,nota_real_mediana
faixa de ganho,,,
menor espaço,25,11.1,80.3
2º quarto,25,24.5,60.2
3º quarto,25,32.4,55.8
maior espaço,25,38.9,48.0


**Lendo as conferências:**

- **(a) As 100 campanhas recebem 3 recomendações:** 81 com 3 melhorias, 15 com 2, 3 com 1 e 1 com nenhuma. Essas últimas já seguem quase tudo e são completadas com "manter".
- **(b) Em 1º lugar aparecem os maiores efeitos:** gancho (50 vezes), rosto (14), muito texto (11), legenda (10).
- **(c) O ganho potencial acompanha a nota real:** Spearman de −0,75. A nota real mediana é 80 no quarto com menos espaço para melhorar e 48 no quarto com mais espaço. Em campanhas que nunca viu, o motor aponta onde está o problema.
- **Isso mostra coerência, não causa:** para provar que seguir a recomendação sobe a nota, seria preciso um teste A/B.

## Resposta à pergunta da Parte 3

> 💡 **Conclusão**
>
> | Requisito | Como o motor atende |
> |---|---|
> | **1. Input e 3 recomendações** | recebe uma linha do dataset original (`recomendar(linha, motor)`, ou `python -m src.recomendacoes KLK-0123`) e devolve sempre 3 (100 de 100 campanhas do teste) |
> | **2. Acionáveis e quantificadas** | cada uma diz o que mudar e quantos pontos de nota (de X para Y), mais o efeito na métrica do objetivo na rede, quando a Parte 1 tem padrão confiável (por exemplo, gancho: +18 pontos e +133% de CTR no LinkedIn) |
> | **3. Contextuais** | só recomenda o que falta no criativo; a plataforma muda o efeito na nota (voz no TikTok) e na métrica (o CTR de cada rede); o objetivo escolhe a métrica; formato só se a rede o usa |
> | **4. Priorização** | em ordem de ganho na nota prevista pelo Modelo A; o que deve ser mantido vem depois |
> | **5. Protótipo funcional** | função Python, linha de comando e este notebook, com autoconferência (`python -m src.recomendacoes`) |
>
> - **Base:** os ganhos vêm do Modelo A da Parte 2 (o simulador "e se") e os efeitos por rede, da Parte 1.
> - **Faz sentido nas campanhas que o motor nunca viu:** as campanhas com mais espaço para melhorar têm nota real menor (Spearman −0,75).
> - **Limite:** é associação, não causa, num dataset simulado. Cada ganho é estimado sozinho; fazer várias mudanças juntas tende a somar, mas a nota não passa de 100.